# Int8 generation

In [1]:
import os
import torch
import numpy as np
OUT_DIR = "../src/test/scala/INT8_test_files"
if not os.path.exists(path=OUT_DIR):
    os.mkdir(OUT_DIR)
    
seed=101
torch.manual_seed(seed=seed)
A = torch.randint(1,10,(4,4))
B = torch.randint(1,10,(4,4))
bias = torch.randint(1,10,(1,4))


X = A@B + bias


np.savetxt(f"{OUT_DIR}/A.txt", A.numpy() , fmt ="%d")
np.savetxt(f"{OUT_DIR}/B.txt", B.numpy(),fmt ="%d")
np.savetxt(f"{OUT_DIR}/bias.txt", bias.numpy(),fmt ="%d")
np.savetxt(f"{OUT_DIR}/X.txt", X.numpy(),fmt ="%d")

In [2]:
import os
import torch
import numpy as np
OUT_DIR = "../src/test/scala/fp16_test_files"
if not os.path.exists(path=OUT_DIR):
    os.mkdir(OUT_DIR)
torch.manual_seed(seed=seed)
A = torch.rand((4,4),dtype=torch.float16)
B = torch.rand((4,4),dtype=torch.float16)
bias = torch.rand((1,4),dtype=torch.float16)


X = A@B + bias


np.savetxt(f"{OUT_DIR}/A.txt", A.numpy() , fmt ="%.4f")
np.savetxt(f"{OUT_DIR}/B.txt", B.numpy(),fmt ="%.4f")
np.savetxt(f"{OUT_DIR}/bias.txt", bias.numpy(),fmt ="%.4f")
np.savetxt(f"{OUT_DIR}/X.txt", X.numpy(),fmt ="%.4f")

In [3]:
import torch
import numpy as np

torch.manual_seed(seed=seed)
M, K, N = 4, 4, 4
OUT_DIR = "../src/test/scala/nf4_test_files"
if not os.path.exists(path=OUT_DIR):
    os.mkdir(OUT_DIR)
NF4_VALS = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0
], dtype=torch.float32)

A = torch.randn(M, K)
B = torch.randn(K, N)

B_indices = torch.zeros(K, N, dtype=torch.int32)
B_scales  = torch.zeros(K, dtype=torch.float32)
B_dq      = torch.zeros(K, N, dtype=torch.float32)

for k in range(K):
    sc = B[k].abs().max().item()
    sc = max(sc, 1e-8)
    B_scales[k] = sc
    for j in range(N):
        idx = (B[k, j].item() / sc - NF4_VALS).abs().argmin().item()
        B_indices[k, j] = idx
        B_dq[k, j] = NF4_VALS[idx].item() * sc

bias = torch.randn(N)

X = A @ B_dq + bias

np.savetxt(f"{OUT_DIR}/A.txt",    A.numpy(),                        fmt="%.6f")
np.savetxt(f"{OUT_DIR}/B.txt",    B_indices.numpy().astype(int),    fmt="%d")
np.savetxt(f"{OUT_DIR}/B_scales.txt", B_scales.numpy().reshape(1, -1),  fmt="%.6f")

np.savetxt(f"{OUT_DIR}/bias.txt", bias.numpy().reshape(1, -1),      fmt="%.6f")
np.savetxt(f"{OUT_DIR}/X.txt",    X.numpy(),                        fmt="%.6f")

# Verify
print("B_indices row 0:", B_indices[0].numpy())
print("B_scales:",        B_scales.numpy())
print("X[0,0]:",          X[0,0].item())

B_indices row 0: [ 0  1 14 13]
B_scales: [1.4567589 1.0674008 0.9191593 1.7717979]
X[0,0]: 4.607222557067871
